# DAFR-Net — Phase 1 Exploration
**K Anirudh | M.Tech AI & DS | SASTRA University**

This notebook covers:
1. MuralDH dataset loading and visualization
2. Damage type distribution analysis
3. Damage classifier training run
4. Dual-branch encoder feature visualization
5. Inpainting results and metrics

In [ ]:
import sys
sys.path.append('..')

import torch
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from PIL import Image
import numpy as np

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1. Dataset overview

In [ ]:
labels = pd.read_csv('../data/damage_labels.csv')
print(labels.head())
print('\nDamage type distribution:')
print(labels['damage_type'].value_counts())

labels['damage_type'].value_counts().plot(kind='bar', color=['#3266ad','#E85D24','#1D9E75'])
plt.title('MuralDH — Damage type distribution')
plt.xlabel('Damage type')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('../results/metrics/damage_distribution.png', dpi=150)
plt.show()

## 2. Sample images by damage type

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, dtype in zip(axes, ['crack', 'fade', 'missing']):
    sample = labels[labels['damage_type'] == dtype].iloc[0]
    img = Image.open(sample['filename'])
    ax.imshow(img)
    ax.set_title(dtype, fontsize=14)
    ax.axis('off')
plt.suptitle('MuralDH — Sample images per damage type', fontsize=16)
plt.tight_layout()
plt.savefig('../results/metrics/sample_images.png', dpi=150)
plt.show()

## 3. Train damage classifier

In [ ]:
import yaml, subprocess
result = subprocess.run(
    ['python', '../src/classifier/train_classifier.py', '--config', '../configs/classifier.yaml'],
    capture_output=True, text=True
)
print(result.stdout[-3000:])   # last 3k chars
print(result.stderr[-1000:])

## 4. Frequency domain visualization

In [ ]:
import torch
import torchvision.transforms as T

sample_path = labels.iloc[0]['filename']
img = Image.open(sample_path).convert('RGB')
t = T.ToTensor()(img).unsqueeze(0)   # (1, 3, 256, 256)

freq = torch.fft.fft2(t, norm='ortho')
freq = torch.fft.fftshift(freq)
magnitude = torch.log(freq.abs() + 1e-8).mean(dim=1).squeeze()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(img)
axes[0].set_title('Original mural')
axes[0].axis('off')
axes[1].imshow(magnitude.numpy(), cmap='inferno')
axes[1].set_title('FFT magnitude spectrum (log scale)')
axes[1].axis('off')
plt.tight_layout()
plt.savefig('../results/metrics/fft_visualization.png', dpi=150)
plt.show()

## 5. Metrics summary
Run this cell after training the encoder + inpainting head.

In [ ]:
import json
metrics_path = Path('../results/metrics/eval_results.json')
if metrics_path.exists():
    with open(metrics_path) as f:
        m = json.load(f)
    print(f"PSNR: {m['psnr']:.2f} dB")
    print(f"SSIM: {m['ssim']:.4f}")
    print(f"Target PSNR > 30 dB: {'✓ PASS' if m['psnr'] > 30 else '✗ FAIL'}")
    print(f"Target SSIM > 0.87: {'✓ PASS' if m['ssim'] > 0.87 else '✗ FAIL'}")
else:
    print('Run training first — results/metrics/eval_results.json not found.')